Notebook to prototype injestion of NYC Building Shape Data, isertion into DuckDB, creating spatial index, and running simple GIS query.


In [ ]:
import duckdb
import polars as pl
from sodapy import Socrata
import requests
from dotenv import load_dotenv
import os

In [8]:
# Load API key from file
SOCRATA_API_KEY = os.getenv('SOCRATA_API_KEY')

# implementation in sodapy

# source domain for NYC Open Data on Socrata
socrata_domain = 'data.cityofnewyork.us'

# initialize client
client = Socrata(
    socrata_domain,
    app_token=SOCRATA_API_KEY,  # Changed from None to api_key
    timeout=100
)

# examine object
print(client)

In [19]:
#Dataset Info:
#Full URL: https://data.cityofnewyork.us/resource/5zhs-2jue.json
nyc_shape_id = '5zhs-2jue'

In [ ]:
def query_data_nycod(dataset_id, custom_limit=None, columns=None, where=None):
    """
    Fetch data from NYC Open Data using sodapy client.
    
    Parameters:
    -----------
    dataset_id : str
        The dataset ID (e.g., '5zhs-2jue')
    custom_limit : int, optional
        Custom row limit. If None, fetches entire dataset.
    columns : list of str or str, optional
        Specific columns to fetch. None or '*' fetches all.
    where : str, optional
        SQL WHERE clause (without the 'WHERE' keyword)
        Example: "bin LIKE '5%'" or "bin IN ('1001', '1002')"
    """
    # Get count
    count_params = {'select': 'COUNT(*)'}
    if where:
        count_params['where'] = where
    
    count_result = client.get(dataset_id, **count_params)
    total_rows = int(count_result[0]['COUNT'])
    
    # Determine limit
    limit = custom_limit if custom_limit else total_rows
    
    # Build query
    query_params = {'limit': limit}
    
    if columns and columns != '*':
        query_params['select'] = ", ".join(columns)
    
    if where:
        query_params['where'] = where
    
    # Fetch and convert
    results = client.get(dataset_id, **query_params)
    return pl.DataFrame(results)

In [ ]:
#Usage example 1
# All BINs starting with 5
df = query_data_nycod(nyc_shape_id, bin_pattern='5%')

# # All BINs starting with 50
# df = query_data_nycod(nyc_shape_id, bin_pattern='50%')

# # BINs ending with 001
# df = query_data_nycod(nyc_shape_id, bin_pattern='%001')

# # BINs containing 123 anywhere
# df = query_data_nycod(nyc_shape_id, bin_pattern='%123%')

# # Combine pattern with specific BINs (OR logic - either pattern OR in list)
# # Note: current implementation uses AND, so you'd want one or the other
# df = query_data_nycod(nyc_shape_id, bin_pattern='5%')